# Tests & Evaluation — Natural Disasters MCP Server

Unit tests for `query_disasters` and evaluation metrics for the chatbot agent.

In [1]:
%pip install pandas mcp[cli] python-dotenv openai langchain langchain-openai -q

Note: you may need to restart the kernel to use updated packages.


In [16]:
import sys
import json
from pathlib import Path

sys.path.insert(0, str(Path("..") / "disasters-server" / "src"))
from disasters_server.server import query_disasters

## Unit Tests — `query_disasters`

In [17]:
passed = 0
failed = 0

def test(name, condition, detail=""):
    global passed, failed
    if condition:
        passed += 1
        print(f"  PASS: {name}")
    else:
        failed += 1
        print(f"  FAIL: {name} -- {detail}")

In [18]:
# Test: query by country
result = await query_disasters(country="Japan", limit=5)
data = json.loads(result)
test("query_by_country returns results", data["total"] > 0)
test("query_by_country all match Japan",
     all(d["Country"] == "Japan" for d in data["disasters"]))

  PASS: query_by_country returns results
  PASS: query_by_country all match Japan


In [19]:
# Test: query by year
result = await query_disasters(year=2010, limit=5)
data = json.loads(result)
test("query_by_year returns results", data["total"] > 0)
test("query_by_year all match 2010",
     all(d["Year"] == 2010 for d in data["disasters"]))

  PASS: query_by_year returns results
  PASS: query_by_year all match 2010


In [20]:
# Test: query by disaster type
result = await query_disasters(disaster_type="Earthquake", limit=5)
data = json.loads(result)
test("query_by_type returns results", data["total"] > 0)
test("query_by_type all match Earthquake",
     all("Earthquake" in d.get("Disaster Type", "") for d in data["disasters"]))

  PASS: query_by_type returns results
  PASS: query_by_type all match Earthquake


In [21]:
# Test: combined filters
result = await query_disasters(country="Colombia", year=2021, disaster_type="Flood", limit=5)
data = json.loads(result)
test("combined_filters returns results", data["total"] > 0)
test("combined_filters country match",
     all(d["Country"] == "Colombia" for d in data["disasters"]))
test("combined_filters year match",
     all(d["Year"] == 2021 for d in data["disasters"]))

  PASS: combined_filters returns results
  PASS: combined_filters country match
  PASS: combined_filters year match


In [22]:
# Test: no results
result = await query_disasters(country="Atlantis", year=9999)
test("no_results returns message", result == "No disasters found matching the criteria.")

  PASS: no_results returns message


In [23]:
# Test: default limit
result = await query_disasters(country="China")
data = json.loads(result)
test("default_limit caps at 10", data["total"] <= 10)

  PASS: default_limit caps at 10


In [24]:
# Test: result structure
result = await query_disasters(country="Japan", year=2011, limit=1)
data = json.loads(result)
disaster = data["disasters"][0]
required_fields = ["Year", "Country", "Disaster Type", "Total Deaths"]
for field in required_fields:
    test(f"result has field '{field}'", field in disaster)

  PASS: result has field 'Year'
  PASS: result has field 'Country'
  PASS: result has field 'Disaster Type'
  PASS: result has field 'Total Deaths'


In [25]:
# Test: app_config validate_env
import os
sys.path.insert(0, str(Path("..") / "streamlit"))
from app_config import validate_env

original = {k: os.environ.get(k) for k in ["OPENAI_API_KEY", "MODEL", "AZURE_ENDPOINT"]}

# Test missing vars
for k in original:
    if k in os.environ:
        del os.environ[k]
ok, msg = validate_env()
test("validate_env fails when vars missing", ok == False)
test("validate_env error mentions missing vars", "Missing" in msg)

# Test present vars
os.environ["OPENAI_API_KEY"] = "test"
os.environ["MODEL"] = "test"
os.environ["AZURE_ENDPOINT"] = "test"
ok, msg = validate_env()
test("validate_env passes when vars present", ok == True)

# Restore original env
for k, v in original.items():
    if v is not None:
        os.environ[k] = v
    elif k in os.environ:
        del os.environ[k]

  PASS: validate_env fails when vars missing
  PASS: validate_env error mentions missing vars
  PASS: validate_env passes when vars present


In [26]:
# Test: conversation helpers
from conversation import format_history, build_user_message

msgs = [{"role": "user", "content": f"msg {i}"} for i in range(10)]
history = format_history(msgs)
test("format_history limits to 6 turns", history.count("msg") == 6)

hm = build_user_message("some history", "my question")
test("build_user_message contains question", "my question" in hm.content)
test("build_user_message contains history", "some history" in hm.content)

  PASS: format_history limits to 6 turns
  PASS: build_user_message contains question
  PASS: build_user_message contains history


In [27]:
print(f"\n{'='*40}")
print(f"RESULTS: {passed} passed, {failed} failed, {passed+failed} total")
print(f"{'='*40}")
assert failed == 0, f"{failed} test(s) failed!"


RESULTS: 21 passed, 0 failed, 21 total


## Evaluation Metrics — Tool Selection Accuracy

We evaluate whether the LLM correctly selects `query_disasters` and extracts the right arguments from natural language queries.

In [31]:
EVAL_DATASET = [
    {"query": "What earthquakes happened in Japan in 2011?",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Japan", "year": "2011", "disaster_type": "Earthquake"}},
    {"query": "Show me floods in Colombia",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Colombia", "disaster_type": "Flood"}},
    {"query": "Hurricanes in the United States in 2005",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "United States", "year": "2005", "disaster_type": "Hurricane"}},
    {"query": "How many people died in earthquakes in Chile?",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Chile", "disaster_type": "Earthquake"}},
    {"query": "Natural disasters in India in 2020",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "India", "year": "2020"}},
    {"query": "Volcanic eruptions in Indonesia",
     "expected_tool": "query_disasters",
     "expected_args": {"country": "Indonesia", "disaster_type": "Volcanic activity"}},
    {"query": "Droughts in Africa",
     "expected_tool": "query_disasters",
     "expected_args": {"disaster_type": "Drought"}},
    {"query": "Tsunamis in 2004",
     "expected_tool": "query_disasters",
     "expected_args": {"year": "2004", "disaster_type": "Earthquake"}},
]

In [32]:
from dotenv import load_dotenv
from openai import AzureOpenAI
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("MODEL")
AZURE_ENDPOINT = os.getenv("AZURE_ENDPOINT")

client = AzureOpenAI(
    api_key=OPENAI_API_KEY,
    api_version="2024-08-01-preview",
    azure_endpoint=AZURE_ENDPOINT,
)


def llm_client(message: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": message},
        ],
    )
    return response.choices[0].message.content


def get_prompt_to_identify_tool_and_arguments(query, tools):
    tools_description = "\n".join(
        [f"- {tool['name']}: {tool['description']} | args: {tool['args']}" for tool in tools]
    )
    return (
        "You are a helpful assistant with access to these tools:\n\n"
        f"{tools_description}\n"
        "Choose the appropriate tool based on the user's question.\n"
        f"User's Question: {query}\n"
        "If no tool is needed, reply directly.\n\n"
        "IMPORTANT: When you need to use a tool, you must ONLY respond with "
        "the exact JSON object format below, nothing else:\n"
        "{\n"
        '    "tool": "tool-name",\n'
        '    "arguments": {\n'
        '        "argument-name": "value"\n'
        "    }\n"
        "}\n\n"
    )


TOOLS = [
    {
        "name": "query_disasters",
        "description": """
            Query the natural disasters CSV dataset.

            Args:
                    country: Filter by country name (case-insensitive). E.g. "Argentina", "Australia". If None, no country filter is applied.
                    year: Filter by year (e.g. 1970). If None, no year filter is applied.
                    disaster_type: Filter by disaster type (case-insensitive). Supported types: "Animal accident", "Drought", "Earthquake", "Epidemic", "Extreme temperature", "Flood", "Fog", "Glacial lake outburst", "Impact", "Insect infestation", "Landslide", "Mass movement (dry)", "Storm", "Volcanic activity", "Wildfire". If None, no disaster type filter is applied.
                    limit: Maximum number of results to return

                Expected output: A JSON string containing a list of disasters matching the criteria, with key details for each disaster. If no disasters match, a message indicating no results found. If the dataset is not loaded, an error message is returned.
                {   "total": 3,
                    "disasters": [
                        {
                        "Year": 2021,
                        "Seq": 182,
                        "Disaster Group": "Natural",
                        "Disaster Subgroup": "Hydrological",
                        "Disaster Type": "Flood",
                        "Country": "Colombia",
                        "ISO": "COL",
                        "Region": "South America",
                        "Continent": "Americas",
                        "Location": "Florencia City (Caquetá Department); Quípama Town (Boyacá Department), Bogotá",
                        "Origin": "Heavy rains",
                        "Associated Dis": "Slide (land, mud, snow, rock)",
                        "Dis Mag Scale": "Km2",
                        "Start Year": 2021,
                        "Start Month": 4.0,
                        "Start Day": 1.0,
                        "End Year": 2021,
                        "End Month": 4.0,
                        "End Day": 5.0,
                        "Total Deaths": 3.0,
                        "No Injured": 5.0,
                        "No Affected": 360.0,
                        "Total Affected": 365.0,
                        "Adm Level": "2",
                        "Admin2 Code": "13608;13691;13914",
                        "Geo Locations": "Florencia, Quipama, Santafe De Bogota D.c. (Adm2). "
                        }
                    ]
                }        
""",
        "args": {
            "country": "str (optional) - filter by country name",
            "year": "int (optional) - filter by year",
            "disaster_type": "str (optional) - filter by disaster type (Flood, Storm, Earthquake, etc.)",
            "limit": "int (default 10) - max results",
        },
    }
]

correct_tool = 0
total_args = 0
correct_args = 0

print("Running evaluation...\n")
for i, entry in enumerate(EVAL_DATASET):
    query = entry["query"]
    prompt = get_prompt_to_identify_tool_and_arguments(query, TOOLS)
    response = llm_client(prompt)

    print(f"[{i+1}/{len(EVAL_DATASET)}] Query: {query}")
    print(f"  LLM response: {response[:200]}")

    try:
        parsed = json.loads(response)
        tool_match = parsed.get("tool") == entry["expected_tool"]
        if tool_match:
            correct_tool += 1
        print(f"  Tool selected: {parsed.get('tool')} {'[correct]' if tool_match else '[WRONG]'}")

        for key, expected_val in entry["expected_args"].items():
            total_args += 1
            actual_val = str(parsed.get("arguments", {}).get(key, ""))
            match = expected_val.lower() in actual_val.lower()
            if match:
                correct_args += 1
            print(f"  Arg '{key}': expected='{expected_val}' got='{actual_val}' {'[correct]' if match else '[WRONG]'}")
    except json.JSONDecodeError:
        print(f"  ERROR: Could not parse JSON from LLM response")

    print()

tool_accuracy = correct_tool / len(EVAL_DATASET) * 100
arg_accuracy = correct_args / total_args * 100 if total_args > 0 else 0

print(f"{'='*50}")
print(f"Tool Selection Accuracy: {correct_tool}/{len(EVAL_DATASET)} ({tool_accuracy:.1f}%)")
print(f"Argument Match Rate:     {correct_args}/{total_args} ({arg_accuracy:.1f}%)")
print(f"{'='*50}")

Running evaluation...



[05/11/26 21:52:45] INFO     HTTP Request: POST                                                     ]8;id=5945954;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=5945955;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://ramiro-bedoya-3347-resource.openai.azure.com/openai/deployment                
                             s/gpt-5-nano/chat/completions?api-version=2024-08-01-preview "HTTP/1.1                
                             200 OK"                                                                               

[1/8] Query: What earthquakes happened in Japan in 2011?
  LLM response: {
  "tool": "query_disasters",
  "arguments": {
    "country": "Japan",
    "year": 2011,
    "disaster_type": "Earthquake",
    "limit": 10
  }
}
  Tool selected: query_disasters [correct]
  Arg 'country': expected='Japan' got='Japan' [correct]
  Arg 'year': expected='2011' got='2011' [correct]
  Arg 'disaster_type': expected='Earthquake' got='Earthquake' [correct]



[05/11/26 21:52:48] INFO     HTTP Request: POST                                                     ]8;id=5945960;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=5945961;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://ramiro-bedoya-3347-resource.openai.azure.com/openai/deployment                
                             s/gpt-5-nano/chat/completions?api-version=2024-08-01-preview "HTTP/1.1                
                             200 OK"                                                                               

[2/8] Query: Show me floods in Colombia
  LLM response: {
  "tool": "query_disasters",
  "arguments": {
    "country": "Colombia",
    "year": null,
    "disaster_type": "Flood",
    "limit": 10
  }
}
  Tool selected: query_disasters [correct]
  Arg 'country': expected='Colombia' got='Colombia' [correct]
  Arg 'disaster_type': expected='Flood' got='Flood' [correct]



...

[6/8] Query: Volcanic eruptions in Indonesia
  LLM response: {
  "tool": "query_disasters",
  "arguments": {
    "country": "Indonesia",
    "year": null,
    "disaster_type": "Volcanic activity",
    "limit": 10
  }
}
  Tool selected: query_disasters [correct]
  Arg 'country': expected='Indonesia' got='Indonesia' [correct]
  Arg 'disaster_type': expected='Volcanic activity' got='Volcanic activity' [correct]



[05/11/26 21:53:15] INFO     HTTP Request: POST                                                     ]8;id=5945990;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=5945991;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://ramiro-bedoya-3347-resource.openai.azure.com/openai/deployment                
                             s/gpt-5-nano/chat/completions?api-version=2024-08-01-preview "HTTP/1.1                
                             200 OK"                                                                               

[7/8] Query: Droughts in Africa
  LLM response: {
  "tool": "query_disasters",
  "arguments": {
    "country": null,
    "year": null,
    "disaster_type": "Drought",
    "limit": 50
  }
}
  Tool selected: query_disasters [correct]
  Arg 'disaster_type': expected='Drought' got='Drought' [correct]



[05/11/26 21:53:22] INFO     HTTP Request: POST                                                     ]8;id=5945996;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py\_client.py]8;;\:]8;id=5945997;file://d:\ws\AI\Final-Practical-Task-chatbot\final-homework\venv\Lib\site-packages\httpx\_client.py#1025\1025]8;;\
                             https://ramiro-bedoya-3347-resource.openai.azure.com/openai/deployment                
                             s/gpt-5-nano/chat/completions?api-version=2024-08-01-preview "HTTP/1.1                
                             200 OK"                                                                               

[8/8] Query: Tsunamis in 2004
  LLM response: {
  "tool": "query_disasters",
  "arguments": {
    "country": null,
    "year": 2004,
    "disaster_type": "Earthquake",
    "limit": 50
  }
}
  Tool selected: query_disasters [correct]
  Arg 'year': expected='2004' got='2004' [correct]
  Arg 'disaster_type': expected='Earthquake' got='Earthquake' [correct]

Tool Selection Accuracy: 8/8 (100.0%)
Argument Match Rate:     16/17 (94.1%)


### Interpretation

- **Tool Selection Accuracy**: Measures whether the LLM correctly identifies `query_disasters` as the tool to use for each query.
- **Argument Match Rate**: Measures whether the extracted arguments (country, year, disaster_type) match the expected values.

A high tool selection accuracy (>90%) indicates the LLM reliably routes disaster-related questions to the correct tool.
Argument match rate reflects the LLM's ability to extract structured parameters from natural language.